# CREATE ML-MODEL

## LOAD DATA

In [71]:
import os
import pandas as pd
import duckdb
import seaborn as sns
import matplotlib.pyplot as plt
from functions import choose_features_target
from sklearn.model_selection import TimeSeriesSplit

os.chdir("/Users/jakoberhard/Library/CloudStorage/GoogleDrive-jakanterh@gmail.com/My Drive/uni/python/TBA_project")

con = duckdb.connect("data/train.duckdb")

con.execute("""
CREATE OR REPLACE TABLE train_delay AS
            SELECT * FROM
            read_parquet('data/train_delay_with_features.parquet')
            """)

df_features = con.execute("SELECT * FROM train_delay").fetchdf()

## PREPROCESSING AND FEATURE SELECTION
Based on EDA in other script

In [72]:
### FUNCTION THAT SPLITS FEATURES AND TARGET
def choose_features_target(df):

    cols_exclude = ["ride_id", # id
                    "delay", # target
                    "departure_real","arrival_real", "departure_planned", "arrival_planned", # raw time variables
                    "train_name", "station_current", "station_start", "station_dest", # high-dimensional categories 
                    "hist_delay_train_q90", "hist_delay_station_q90", "stops_total", "stop_index", "share_ride_time" # discarded after correlation analysis
                    ]
    
    feature_cols = [col for col in df.columns if col not in cols_exclude]
    X = df[feature_cols]
    y = df["delay"]

    return X, y

# separate features and target
X, y = choose_features_target(df_features)

In [73]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# define features groups
feature_scheme = {
    "categorical_one_hot": [
        "train_type", "station_role" # low number of dimensions (hot encoding ok)
    ],
    "numeric_scaled": [
        "temperature", "precipitation_log", "wind_speed", 
        "hour_sin", "hour_cos", "weekday_sin", "weekday_cos", 
        "month_sin", "month_cos", "travel_time", "dwell_time_planned", 
        "time_since_start_planned",
        "hist_delay_station_avg", "hist_delay_station_count",
        "hist_delay_train_avg", "hist_delay_train_count"
    ],
    "binary_passthrough": [
        "precipitation_any", "feast", "weekend", 
        "departure_rush_morning", "departure_rush_evening"
    ]
}


### PREPROCESSOR ###

preprocessor = ColumnTransformer(
    transformers=[

        # categories: one-hot 
        ("cat", OneHotEncoder(handle_unknown = "ignore", sparse_output = False), feature_scheme["categorical_one_hot"]),
        # numeric variables: standard scaling
        ("num", StandardScaler(), feature_scheme["numeric_scaled"]),
        # binary variables: do nothing
        ("pass", "passthrough", feature_scheme["binary_passthrough"])
    ],
    # drop rest
    remainder = "drop")

## CROSS-VALIDATION
### DETERMINE GAP BETWEEN FOLDS

In [74]:
# get count of total observed hours and rows per hour (average)
total_hours = (df_features["departure_real"].max() - df_features["arrival_real"].min()).total_seconds() / 3600
rows_per_hour = len(df_features) / total_hours

# define number of hours that should be in between of folds
min_hours = 5

# calculate gap: rows per hour times gap
recommended_gap = int(rows_per_hour * min_hours)

print(f"Rows per hour: {rows_per_hour:.1f}")
print(f"Recommended minimal gap: {recommended_gap}")

Rows per hour: 139.8
Recommended minimal gap: 699


### DEFINE TIME SERIES CROSS-VALIDATOR

In [75]:
ts_cv = TimeSeriesSplit(
    n_splits = 5,
    # lets be conservative: 
    # in very busy times, lower gap number might cause problems
    gap = 1000
)

In [83]:
# select data for training, tuning and testing
X_tune = X.iloc[-600_000:-300_000]
y_tune = y.iloc[-600_000:-300_000]

X_train = X.iloc[:-300_000]
y_train = y.iloc[:-300_000]

X_test = X.iloc[-300_000:]
y_test = y.iloc[-300_000:]

## TUNING THE MODELS

### PIPELINE

In [80]:
# build pipelines for all models 


from sklearn_quantile import RandomForestQuantileRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline



pipe_rf_tune = make_pipeline(
    preprocessor,
    RandomForestQuantileRegressor(
        random_state = 42,
        q=[0.05, 0.5, 0.95], 
    )
)

pipe_hgb_mean_tune = make_pipeline(
    preprocessor,
    HistGradientBoostingRegressor(
        loss = "squared_error",
        random_state = 42,
    )
)

# pipe_hgb_q05 = make_pipeline(
#     preprocessor,
#     HistGradientBoostingRegressor(
#         loss = "quantile",
#         quantile = 0.05,
#         random_state = 42,
#     )
# )

# pipe_hgb_q95 = make_pipeline(
#     preprocessor,
#     HistGradientBoostingRegressor(
#         loss = "quantile",
#         quantile = 0.95,
#         random_state = 42,
#     )
# )

### HYPERPARAMETERS

In [81]:
from scipy.stats import loguniform, randint

# HistGradientBoosting Space
hyp_hgb = {
    "histgradientboostingregressor__learning_rate": loguniform(0.01, 0.2), 
    "histgradientboostingregressor__max_iter": randint(100, 500),        
    "histgradientboostingregressor__max_leaf_nodes": randint(15, 60),    
    "histgradientboostingregressor__min_samples_leaf": randint(20, 100),
}

# Random Forest Space
hyp_rf = {
    "randomforestquantileregressor__n_estimators": randint(100, 500),
    "randomforestquantileregressor__max_depth": randint(5, 30),
    "randomforestquantileregressor__min_samples_leaf": randint(10, 100),
}

In [82]:
# initialize randomizedsearchcv

from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, mean_pinball_loss


search_hgb = RandomizedSearchCV(
    pipe_hgb_mean_tune,
    param_distributions = hyp_hgb,
    n_iter = 15,               
    scoring = "neg_mean_absolute_error",
    cv = ts_cv,
    n_jobs = -1,
    verbose = 1,
    random_state = 42,)


# # make scorer
# pinball_05 = make_scorer(mean_pinball_loss, greater_is_better = False, alpha = 0.05,)
# pinball_95 = make_scorer(mean_pinball_loss, greater_is_better = False, alpha = 0.95,)

# search_hgb_q05 = RandomizedSearchCV(
#     pipe_q05,
#     param_distributions=hyp_hgb,
#     n_iter=10,
#     scoring=pinball_05,
#     cv=ts_cv,
#     n_jobs=-1,
#     verbose=1,
#     random_state=42,
# )

# search_hgb_q95 = RandomizedSearchCV(
#     pipe_hgb_q95,
#     param_distributions=hyp_hgb,
#     n_iter=10,
#     scoring=pinball_95,
#     cv=ts_cv,
#     n_jobs=-1,
#     verbose=1,
#     random_state=42,
# )


# add rf
def rf_median_mae(y_true, y_pred):
    median_preds = y_pred[:, 1]
    return -np.mean(np.abs(y_true - median_preds))

rf_scorer = make_scorer(rf_median_mae)

search_rf = RandomizedSearchCV(
    pipe_rf_tune,
    param_distributions = hyp_rf,
    n_iter = 10,
    scoring = rf_scorer, 
    cv = ts_cv,
    n_jobs = -1,
    verbose = 1,
    random_state = 42)


In [ ]:
search_hgb.fit(X_tune, y_tune)
best_hgb_params = search_hgb.best_params_

import gc
del search_hgb
gc.collect()

## TRAINING

In [ ]:
pipe_hgb_mean = make_pipeline(
    preprocessor,
    HistGradientBoostingRegressor(
        loss = "squared_error",
        random_state = 42,)).set_params(**best_hgb_params)

pipe_hgb_q05 = make_pipeline(
    preprocessor,
    HistGradientBoostingRegressor(
        loss = "quantile",
        quantile = 0.05,
        random_state = 42,)).set_params(**best_hgb_params)

pipe_hgb_q95 = make_pipeline(
    preprocessor,
    HistGradientBoostingRegressor(
        loss = "quantile",
        quantile = 0.95,
        random_state = 42,)).set_params(**best_hgb_params)

pipe_hgb_mean.fit(X_train, y_train)
pipe_hgb_q05.fit(X_train, y_train)
pipe_hgb_q95.fit(X_train, y_train)

In [ ]:
search_rf.fit(X_tune, y_tune)
best_rf_params = search_rf.best_params_

pipe_rf = make_pipeline(
    preprocessor,
    RandomForestQuantileRegressor(
        random_state = 42,
        q=[0.05, 0.5, 0.95], )).set_params(**best_rf_params)

pipe_rf.fit(X_train, y_train)

## TEST

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_pinball_loss

def evaluate_interval_model(model_50, model_05, model_95, X_test, y_test):

    y_pred_50 = model_50.predict(X_test)
    y_pred_05 = model_05.predict(X_test)
    y_pred_95 = model_95.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred_50)

    pinball_05 = mean_pinball_loss(y_test, y_pred_05, alpha = 0.05)
    pinball_95 = mean_pinball_loss(y_test, y_pred_95, alpha = 0.95)

    coverage = np.mean(
        (y_test >= y_pred_05) & (y_test <= y_pred_95)
    )

    interval_width = np.mean(y_pred_95 - y_pred_05)

    return {
        "MAE": mae,
        "Pinball_05": pinball_05,
        "Pinball_95": pinball_95,
        "Coverage_90_interval": coverage,
        "Avg_Interval_Width": interval_width,
    }


In [ ]:
results_hgb = evaluate_interval_model(
    pipe_hgb_mean,
    pipe_hgb_q05,
    pipe_hgb_q95,
    X_test,
    y_test)

# RF evaluation
rf_preds = pipe_rf.predict(X_test)
rf_lo  = rf_preds[:, 0]
rf_med = rf_preds[:, 1]
rf_hi  = rf_preds[:, 2]

results_rf = {
    "MAE": mean_absolute_error(y_test, rf_med),
    "Pinball_05": mean_pinball_loss(y_test, rf_lo, alpha=0.05),
    "Pinball_95": mean_pinball_loss(y_test, rf_hi, alpha=0.95),
    "Coverage_90_interval": np.mean((y_test >= rf_lo) & (y_test <= rf_hi)),
    "Avg_Interval_Width": np.mean(rf_hi - rf_lo),
}


print("HGB Results:", results_hgb)
print("RF Results:", results_rf)


In [ ]:
### OLD ###


### EVALUATE MODEL ###
from sklearn.model_selection import cross_validate

def evaluate(pipeline, X, y, cv):
    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring={
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error",
        },
        n_jobs=-1,
    )

    mae = -scores["test_mae"]
    rmse = -scores["test_rmse"]

    return {
    "MAE_mean": float(mae.mean()),
    "MAE_std": float(mae.std()),
    "RMSE_mean": float(rmse.mean()),
    "RMSE_std": float(rmse.std()),
    }



In [23]:
evaluate(pipeline_test, X, y, cv=ts_cv)


{'MAE_mean': 9.272330174183098,
 'MAE_std': 0.5352379500782578,
 'RMSE_mean': 15.342233401211113,
 'RMSE_std': 0.9148329932764727}

## SAVE MODEL

In [22]:
import joblib

# Suppose you already fitted your pipeline
pipeline_first.fit(X, y["delay"])

# Export to a file
joblib.dump(pipeline_first, "trained_delay_model.pkl")
print("Pipeline saved as 'trained_delay_model.pkl'")


Pipeline saved as 'trained_delay_model.pkl'
